# 04 Uplift Modeling

This notebook builds and evaluates uplift-style targeting analyses using processed session and customer tables.

**Expected processed inputs**
- `session_level_funnel.csv`
- `customer_level_features.csv`
- `event_master.csv`

**Main exported summaries**
- decile and cumulative uplift reports
- Qini / AUUC summary tables
- bootstrap CI tables
- campaign incrementality summaries
- strategy tables


In [ ]:
# ================================================================
# PART 4B-1. Uplift baseline (revised)
# - Goal: identify sessions with high incremental purchase uplift when showing Variant_B
# - Method: T-learner (one Control model + one Variant_B model)
# - Main improvements:
#   1) Use core customer profile fields + signup_date
#   2) Build past_* history features using only information before session_date
#   3) Add campaign metadata (channel/objective/target_segment/expected_uplift)
# ================================================================
import os
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import HistGradientBoostingClassifier

# ------------------------------------------------
# 0) Set paths
# ------------------------------------------------
BASE_PATH = r"../data/processed"

session_path = os.path.join(BASE_PATH, "session_level_funnel.csv")
customer_path = os.path.join(BASE_PATH, "customer_level_features.csv")
event_master_path = os.path.join(BASE_PATH, "event_master.csv")

# ------------------------------------------------
# 1) Load data
# ------------------------------------------------
sdf = pd.read_csv(session_path)
cdf = pd.read_csv(customer_path)
em = pd.read_csv(
    event_master_path,
    usecols=lambda c: c in ["campaign_id", "channel", "objective", "target_segment", "expected_uplift"]
)

print("✅ Data loaded successfully")
print("session_level_funnel:", sdf.shape)
print("customer_level_features:", cdf.shape)
print("event_master(meta):", em.shape)

# ------------------------------------------------
# 2) Keep only Variant_B vs Control
# ------------------------------------------------
df = sdf[sdf["experiment_group"].isin(["Control", "Variant_B"])].copy()

df["treatment"] = np.where(df["experiment_group"] == "Variant_B", 1, 0)
df["y"] = pd.to_numeric(df["any_purchase"], errors="coerce").fillna(0).astype(int)

print("\n✅ Rows used for analysis:", len(df))
print(df["experiment_group"].value_counts(dropna=False))
print("purchase mean:", df["y"].mean())

# ------------------------------------------------
# 3) Merge base customer profile fields
# - Avoid final aggregate features (such as total_orders) to reduce leakage
# ------------------------------------------------
base_customer_cols = [
    "customer_id",
    "signup_date",
    "country",
    "age",
    "gender",
    "loyalty_tier",
    "acquisition_channel"
]
base_customer_cols = [c for c in base_customer_cols if c in cdf.columns]

cust_base = cdf[base_customer_cols].drop_duplicates(subset=["customer_id"]).copy()
df = df.merge(cust_base, on="customer_id", how="left")

# ------------------------------------------------
# 4) Merge campaign metadata
# ------------------------------------------------
camp_meta = em.drop_duplicates(subset=["campaign_id"]).copy()
camp_meta["campaign_id"] = pd.to_numeric(camp_meta["campaign_id"], errors="coerce").fillna(0).astype(int)

df["session_campaign_id"] = pd.to_numeric(df["session_campaign_id"], errors="coerce").fillna(0).astype(int)
df = df.merge(
    camp_meta,
    left_on="session_campaign_id",
    right_on="campaign_id",
    how="left"
)

# ------------------------------------------------
# 5) Clean date columns
# ------------------------------------------------
df["session_date"] = pd.to_datetime(df["session_date"], errors="coerce")
df["signup_date"] = pd.to_datetime(df["signup_date"], errors="coerce")

# Drop rows with missing session_date
df = df[~df["session_date"].isna()].copy()

# ------------------------------------------------
# 6) Create point-in-time history features
# - Key rule: use only information available before the current session
# ------------------------------------------------
# Sorting is important
df = df.sort_values(["customer_id", "session_date", "session_id"]).reset_index(drop=True)

# Fix numeric columns
base_num_cols = [
    "any_view", "any_click", "any_add_to_cart", "any_purchase",
    "is_campaign_exposed_session"
]
for c in base_num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# Group by customer
g = df.groupby("customer_id", sort=False)

# Past session count
df["past_total_sessions"] = g.cumcount()

# Cumulative past behavior counts (exclude the current session)
df["past_total_views"] = g["any_view"].cumsum() - df["any_view"]
df["past_total_clicks"] = g["any_click"].cumsum() - df["any_click"]
df["past_total_carts"] = g["any_add_to_cart"].cumsum() - df["any_add_to_cart"]
df["past_total_purchases"] = g["any_purchase"].cumsum() - df["any_purchase"]
df["past_campaign_exposed_sessions"] = (
    g["is_campaign_exposed_session"].cumsum() - df["is_campaign_exposed_session"]
)

# Past cart-abandonment session count
df["cart_abandon_current"] = np.where(
    (df["any_add_to_cart"] == 1) & (df["any_purchase"] == 0),
    1, 0
)
df["past_cart_abandon_sessions"] = g["cart_abandon_current"].cumsum() - df["cart_abandon_current"]

# Past purchase-history flag
df["past_has_purchase_history"] = (df["past_total_purchases"] > 0).astype(int)

# Ratio-style past-history features
df["past_session_conversion_rate"] = np.where(
    df["past_total_sessions"] > 0,
    df["past_total_purchases"] / df["past_total_sessions"],
    0.0
)

df["past_view_to_click_rate"] = np.where(
    df["past_total_views"] > 0,
    df["past_total_clicks"] / df["past_total_views"],
    0.0
)

df["past_click_to_cart_rate"] = np.where(
    df["past_total_clicks"] > 0,
    df["past_total_carts"] / df["past_total_clicks"],
    0.0
)

df["past_cart_to_purchase_rate"] = np.where(
    df["past_total_carts"] > 0,
    df["past_total_purchases"] / df["past_total_carts"],
    0.0
)

# Gap since the previous session
df["prev_session_date"] = g["session_date"].shift(1)
df["past_days_since_last_session"] = (
    (df["session_date"] - df["prev_session_date"]).dt.days
)
df["past_days_since_last_session"] = df["past_days_since_last_session"].fillna(-1)

# Gap since the previous purchase date
purchase_date = df["session_date"].where(df["any_purchase"] == 1)
last_purchase_inclusive = purchase_date.groupby(df["customer_id"]).ffill()
df["last_purchase_before_session"] = last_purchase_inclusive.groupby(df["customer_id"]).shift(1)

df["past_days_since_last_purchase"] = (
    (df["session_date"] - df["last_purchase_before_session"]).dt.days
)
df["past_days_since_last_purchase"] = df["past_days_since_last_purchase"].fillna(-1)

# Days since signup
df["customer_tenure_days_sofar"] = (df["session_date"] - df["signup_date"]).dt.days
# Prevent synthetic anomalies
df["customer_tenure_days_sofar"] = df["customer_tenure_days_sofar"].clip(lower=0)
df["customer_tenure_days_sofar"] = df["customer_tenure_days_sofar"].fillna(0)

# Date-derived features
df["session_month"] = df["session_date"].dt.month.fillna(0).astype(int).astype(str)
df["session_dayofweek"] = df["session_date"].dt.dayofweek.fillna(0).astype(int).astype(str)

# ------------------------------------------------
# 7) Build the feature set
# - Safe base profile + session-start context + past history
# ------------------------------------------------
categorical_features = [
    "country",
    "gender",
    "loyalty_tier",
    "acquisition_channel",
    "first_device_type",
    "first_traffic_source",
    "first_page_category",
    "channel",
    "objective",
    "target_segment",
    "session_campaign_id",
    "session_month",
    "session_dayofweek"
]

numeric_features = [
    "age",
    "expected_uplift",
    "customer_tenure_days_sofar",
    "past_total_sessions",
    "past_total_views",
    "past_total_clicks",
    "past_total_carts",
    "past_total_purchases",
    "past_campaign_exposed_sessions",
    "past_cart_abandon_sessions",
    "past_has_purchase_history",
    "past_session_conversion_rate",
    "past_view_to_click_rate",
    "past_click_to_cart_rate",
    "past_cart_to_purchase_rate",
    "past_days_since_last_session",
    "past_days_since_last_purchase"
]

categorical_features = [c for c in categorical_features if c in df.columns]
numeric_features = [c for c in numeric_features if c in df.columns]
features = categorical_features + numeric_features

# Handle missing values
for c in categorical_features:
    df[c] = df[c].fillna("Unknown").astype(str)

for c in numeric_features:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

# Use session_campaign_id as a categorical feature
if "session_campaign_id" in df.columns:
    df["session_campaign_id"] = df["session_campaign_id"].astype(int).astype(str)

print("\n✅ Number of features used:", len(features))
print(features)

# ------------------------------------------------
# 8) Optional sampling
# - Build history features on the full dataset before sampling
# ------------------------------------------------
MAX_ROWS = 500000
RANDOM_STATE = 42

if MAX_ROWS is not None and len(df) > MAX_ROWS:
    df["strata"] = df["treatment"].astype(str) + "_" + df["y"].astype(str)
    df = (
        df.groupby("strata", group_keys=False)
          .apply(lambda x: x.sample(
              n=max(1, int(len(x) / len(df) * MAX_ROWS)),
              random_state=RANDOM_STATE
          ))
          .reset_index(drop=True)
    )
    print(f"\n✅ Rows after sampling: {len(df):,}")
else:
    print(f"\n✅ Using full dataset rows: {len(df):,}")

# ------------------------------------------------
# 9) train / valid split
# ------------------------------------------------
df["strata"] = df["treatment"].astype(str) + "_" + df["y"].astype(str)

train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df["strata"]
)

print("\n✅ train / valid split")
print("train:", train_df.shape, "valid:", valid_df.shape)

# ------------------------------------------------
# 10) one-hot encoding
# ------------------------------------------------
X_train_raw = train_df[features].copy()
X_valid_raw = valid_df[features].copy()

all_X = pd.concat([X_train_raw, X_valid_raw], axis=0)
all_X = pd.get_dummies(all_X, drop_first=False)

X_train = all_X.iloc[:len(X_train_raw)].copy()
X_valid = all_X.iloc[len(X_train_raw):].copy()

y_train = train_df["y"].values
y_valid = valid_df["y"].values
t_train = train_df["treatment"].values
t_valid = valid_df["treatment"].values

X_train_t = X_train[t_train == 1]
y_train_t = y_train[t_train == 1]

X_train_c = X_train[t_train == 0]
y_train_c = y_train[t_train == 0]

print("\n✅ train treated / control")
print("treated:", X_train_t.shape, "control:", X_train_c.shape)

# ------------------------------------------------
# 11) Define models
# ------------------------------------------------
use_xgb = False
try:
    from xgboost import XGBClassifier
    use_xgb = True
except Exception:
    use_xgb = False

def make_model():
    if use_xgb:
        return XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    else:
        return HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_depth=6,
            max_iter=300,
            random_state=RANDOM_STATE
        )

model_t = make_model()
model_c = make_model()

print("\n✅ model type:", "XGBoost" if use_xgb else "HistGradientBoosting")

# ------------------------------------------------
# 12) Train the T-learner
# ------------------------------------------------
model_t.fit(X_train_t, y_train_t)
model_c.fit(X_train_c, y_train_c)

p_t_valid = model_t.predict_proba(X_valid)[:, 1]
p_c_valid = model_c.predict_proba(X_valid)[:, 1]
uplift_valid = p_t_valid - p_c_valid

auc_t = roc_auc_score(y_valid[t_valid == 1], p_t_valid[t_valid == 1]) if len(np.unique(y_valid[t_valid == 1])) > 1 else np.nan
auc_c = roc_auc_score(y_valid[t_valid == 0], p_c_valid[t_valid == 0]) if len(np.unique(y_valid[t_valid == 0])) > 1 else np.nan

print("\n==================== Model Performance (Individual Response Models) ====================")
print(f"Treated model AUC  : {auc_t:.4f}")
print(f"Control model AUC  : {auc_c:.4f}")

# ------------------------------------------------
# 13) validation uplift decile report
# ------------------------------------------------
valid_eval = valid_df[[
    "customer_id", "session_id", "experiment_group", "treatment", "y"
]].copy()

valid_eval["p_treat"] = p_t_valid
valid_eval["p_control"] = p_c_valid
valid_eval["uplift_score"] = uplift_valid

valid_eval["uplift_decile"] = pd.qcut(
    -valid_eval["uplift_score"].rank(method="first"),
    q=10,
    labels=False
) + 1

decile_report = (
    valid_eval.groupby("uplift_decile")
              .apply(lambda x: pd.Series({
                  "n": len(x),
                  "treated_n": (x["treatment"] == 1).sum(),
                  "control_n": (x["treatment"] == 0).sum(),
                  "treated_purchase_rate": x.loc[x["treatment"] == 1, "y"].mean() if (x["treatment"] == 1).sum() > 0 else np.nan,
                  "control_purchase_rate": x.loc[x["treatment"] == 0, "y"].mean() if (x["treatment"] == 0).sum() > 0 else np.nan,
                  "observed_uplift": (
                      x.loc[x["treatment"] == 1, "y"].mean() - x.loc[x["treatment"] == 0, "y"].mean()
                  ) if ((x["treatment"] == 1).sum() > 0 and (x["treatment"] == 0).sum() > 0) else np.nan,
                  "avg_predicted_uplift": x["uplift_score"].mean()
              }))
              .reset_index()
              .sort_values("uplift_decile")
)

print("\n==================== Validation Uplift Decile Report ====================")
display(decile_report)

cum_rows = []
for d in range(1, 11):
    tmp = valid_eval[valid_eval["uplift_decile"] <= d].copy()
    tr = tmp[tmp["treatment"] == 1]["y"].mean() if (tmp["treatment"] == 1).sum() > 0 else np.nan
    ct = tmp[tmp["treatment"] == 0]["y"].mean() if (tmp["treatment"] == 0).sum() > 0 else np.nan
    cum_rows.append({
        "top_deciles_included": d,
        "n": len(tmp),
        "treated_purchase_rate": tr,
        "control_purchase_rate": ct,
        "observed_uplift": tr - ct if pd.notna(tr) and pd.notna(ct) else np.nan
    })

cum_report = pd.DataFrame(cum_rows)

print("\n==================== Cumulative Top-Decile Report ====================")
display(cum_report)

# ------------------------------------------------
# 14) Score all sessions with uplift
# ------------------------------------------------
full_X_raw = df[features].copy()
full_X = pd.get_dummies(full_X_raw, drop_first=False)
full_X = full_X.reindex(columns=X_train.columns, fill_value=0)

p_t_full = model_t.predict_proba(full_X)[:, 1]
p_c_full = model_c.predict_proba(full_X)[:, 1]
uplift_full = p_t_full - p_c_full

scored = df[[
    "customer_id", "session_id", "experiment_group", "treatment", "y"
]].copy()

for c in [
    "first_traffic_source", "first_device_type", "first_page_category",
    "session_campaign_id", "channel", "objective", "target_segment",
    "country", "age", "gender", "loyalty_tier", "acquisition_channel",
    "past_total_sessions", "past_total_purchases", "past_session_conversion_rate",
    "past_cart_abandon_sessions"
]:
    if c in df.columns:
        scored[c] = df[c]

scored["pred_purchase_prob_if_B"] = p_t_full
scored["pred_purchase_prob_if_Control"] = p_c_full
scored["pred_uplift_B_vs_Control"] = uplift_full

scored = scored.sort_values("pred_uplift_B_vs_Control", ascending=False).reset_index(drop=True)

cutoff = scored["pred_uplift_B_vs_Control"].quantile(0.80)
scored["target_flag_top20pct"] = (scored["pred_uplift_B_vs_Control"] >= cutoff).astype(int)

print("\n==================== Example Top Targets ====================")
display(scored.head(20))

# ------------------------------------------------
# 15) Save outputs
# ------------------------------------------------
decile_path = os.path.join(BASE_PATH, "part4b_variantB_uplift_decile_report_v2.csv")
cum_path = os.path.join(BASE_PATH, "part4b_variantB_uplift_cumulative_report_v2.csv")
scored_path = os.path.join(BASE_PATH, "part4b_variantB_scored_sessions_v2.csv")
top_targets_path = os.path.join(BASE_PATH, "part4b_variantB_top20_targets_v2.csv")

decile_report.to_csv(decile_path, index=False)
cum_report.to_csv(cum_path, index=False)
scored.to_csv(scored_path, index=False)
scored[scored["target_flag_top20pct"] == 1].to_csv(top_targets_path, index=False)

print("\n✅ PART 4B-1 revised version complete")
print("Saved files:")
print("-", decile_path)
print("-", cum_path)
print("-", scored_path)
print("-", top_targets_path)

# ------------------------------------------------
# 16) One-line interpretation
# ------------------------------------------------
best_decile = decile_report.sort_values("observed_uplift", ascending=False).head(1)

print("\n==================== Interpretation Summary ====================")
if len(best_decile) == 1:
    d = best_decile.iloc[0]
    print(
        f"The strongest uplift decile is decile {int(d['uplift_decile'])} "
        f"with observed uplift {d['observed_uplift']:.4f}, "
        f"and average predicted uplift {d['avg_predicted_uplift']:.4f}."
    )
else:
    print("Please check the decile report.")

In [ ]:
# ================================================================
# PART 4B-Tree. Interpretable Uplift Tree (Transformed Outcome Tree)
# Assumptions:
# - df: dataframe after v2 feature engineering
# - features: list of pre-treatment features to use
# - BASE_PATH: save path
# Goal:
# - "under which conditions Variant_B uplift is especially large
#   in the form of leaf-level (segment) rules
# ================================================================
import os
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

# ------------------------------------------------
# 0) Check assumptions
# ------------------------------------------------
required_vars = ["df", "features", "BASE_PATH"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise ValueError(f"Please run the v2 feature-engineering cell first. Missing variables: {missing_vars}")

work_df = df.copy()

# Check treatment / outcome
if "treatment" not in work_df.columns:
    work_df["treatment"] = np.where(work_df["experiment_group"] == "Variant_B", 1, 0)
if "y" not in work_df.columns:
    work_df["y"] = pd.to_numeric(work_df["any_purchase"], errors="coerce").fillna(0).astype(int)

# ------------------------------------------------
# 1) Sampling (to manage runtime if the data is too large)
# ------------------------------------------------
MAX_ROWS = 500000
RANDOM_STATE = 42

if MAX_ROWS is not None and len(work_df) > MAX_ROWS:
    work_df["strata"] = work_df["treatment"].astype(str) + "_" + work_df["y"].astype(str)
    work_df = (
        work_df.groupby("strata", group_keys=False)
               .apply(lambda x: x.sample(
                   n=max(1, int(len(x) / len(work_df) * MAX_ROWS)),
                   random_state=RANDOM_STATE
               ))
               .reset_index(drop=True)
    )
    print(f"✅ Rows after sampling: {len(work_df):,}")
else:
    print(f"✅ Using full dataset rows: {len(work_df):,}")

# ------------------------------------------------
# 2) train / valid split
# ------------------------------------------------
work_df["strata"] = work_df["treatment"].astype(str) + "_" + work_df["y"].astype(str)

train_df, valid_df = train_test_split(
    work_df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=work_df["strata"]
)

print("\n✅ train / valid split")
print("train:", train_df.shape, "valid:", valid_df.shape)

# ------------------------------------------------
# 3) One-hot encoding (pre-treatment features only)
# ------------------------------------------------
X_train_raw = train_df[features].copy()
X_valid_raw = valid_df[features].copy()

all_X = pd.concat([X_train_raw, X_valid_raw], axis=0)
all_X = pd.get_dummies(all_X, drop_first=False)

X_train = all_X.iloc[:len(X_train_raw)].copy()
X_valid = all_X.iloc[len(X_train_raw):].copy()

y_train = pd.to_numeric(train_df["y"], errors="coerce").fillna(0).astype(int).values
y_valid = pd.to_numeric(valid_df["y"], errors="coerce").fillna(0).astype(int).values
t_train = pd.to_numeric(train_df["treatment"], errors="coerce").fillna(0).astype(int).values
t_valid = pd.to_numeric(valid_df["treatment"], errors="coerce").fillna(0).astype(int).values

# ------------------------------------------------
# 4) Create the transformed outcome
# Because the experiment is randomized, propensity p is treated as constant
# Z = Y*T/p - Y*(1-T)/(1-p)
# E[Z|X] = uplift tau(X)
# ------------------------------------------------
p = t_train.mean()
if p <= 0 or p >= 1:
    raise ValueError("The treatment share is too close to 0 or 1.")

z_train = y_train * (t_train / p) - y_train * ((1 - t_train) / (1 - p))

print(f"\n✅ train propensity p = {p:.4f}")
print(f"✅ transformed outcome mean = {z_train.mean():.6f}")

# ------------------------------------------------
# 5) Train a shallow interpretable tree
# - Keep max_depth shallow for interpretability
# - Use a large min_samples_leaf so leaf-level uplift is more stable
# ------------------------------------------------
tree_model = DecisionTreeRegressor(
    max_depth=3,           # 3-4 is a good starting range
    min_samples_leaf=3000, # too small makes leaf-level uplift unstable
    random_state=RANDOM_STATE
)

tree_model.fit(X_train, z_train)

# validation uplift score
uplift_valid = tree_model.predict(X_valid)

# ------------------------------------------------
# 6) validation decile report
# ------------------------------------------------
valid_eval = valid_df[[
    "customer_id", "session_id", "experiment_group", "treatment", "y"
]].copy()

valid_eval["uplift_score"] = uplift_valid
valid_eval["uplift_decile"] = pd.qcut(
    -valid_eval["uplift_score"].rank(method="first"),
    q=10,
    labels=False
) + 1

decile_report_tree = (
    valid_eval.groupby("uplift_decile")
              .apply(lambda x: pd.Series({
                  "n": len(x),
                  "treated_n": (x["treatment"] == 1).sum(),
                  "control_n": (x["treatment"] == 0).sum(),
                  "treated_purchase_rate": x.loc[x["treatment"] == 1, "y"].mean() if (x["treatment"] == 1).sum() > 0 else np.nan,
                  "control_purchase_rate": x.loc[x["treatment"] == 0, "y"].mean() if (x["treatment"] == 0).sum() > 0 else np.nan,
                  "observed_uplift": (
                      x.loc[x["treatment"] == 1, "y"].mean() - x.loc[x["treatment"] == 0, "y"].mean()
                  ) if ((x["treatment"] == 1).sum() > 0 and (x["treatment"] == 0).sum() > 0) else np.nan,
                  "avg_predicted_uplift": x["uplift_score"].mean()
              }))
              .reset_index()
              .sort_values("uplift_decile")
)

print("\n==================== Uplift Tree Validation Decile Report ====================")
display(decile_report_tree)

# cumulative report
cum_rows = []
for d in range(1, 11):
    tmp = valid_eval[valid_eval["uplift_decile"] <= d].copy()
    tr = tmp[tmp["treatment"] == 1]["y"].mean() if (tmp["treatment"] == 1).sum() > 0 else np.nan
    ct = tmp[tmp["treatment"] == 0]["y"].mean() if (tmp["treatment"] == 0).sum() > 0 else np.nan
    cum_rows.append({
        "top_deciles_included": d,
        "n": len(tmp),
        "treated_purchase_rate": tr,
        "control_purchase_rate": ct,
        "observed_uplift": tr - ct if pd.notna(tr) and pd.notna(ct) else np.nan
    })

cum_report_tree = pd.DataFrame(cum_rows)

print("\n==================== Uplift Tree Cumulative Top-Decile Report ====================")
display(cum_report_tree)

# ------------------------------------------------
# 7) Assign leaf IDs
# ------------------------------------------------
leaf_valid = tree_model.apply(X_valid)
valid_eval["leaf_id"] = leaf_valid

full_X_raw = work_df[features].copy()
full_X = pd.get_dummies(full_X_raw, drop_first=False)
full_X = full_X.reindex(columns=X_train.columns, fill_value=0)

work_df["uplift_score_tree"] = tree_model.predict(full_X)
work_df["leaf_id"] = tree_model.apply(full_X)

# ------------------------------------------------
# 8) Extract leaf rules
# ------------------------------------------------
feature_names = list(X_train.columns)
tree_ = tree_model.tree_

def get_leaf_rules(tree_model, feature_names):
    tree_ = tree_model.tree_
    rules = {}

    def recurse(node, path):
        feature_idx = tree_.feature[node]
        threshold = tree_.threshold[node]

        # leaf
        if feature_idx == -2:
            rules[node] = " AND ".join(path) if path else "ROOT"
            return

        feature_name = feature_names[feature_idx]

        left_path = path + [f"({feature_name} <= {threshold:.4f})"]
        recurse(tree_.children_left[node], left_path)

        right_path = path + [f"({feature_name} > {threshold:.4f})"]
        recurse(tree_.children_right[node], right_path)

    recurse(0, [])
    return rules

leaf_rule_map = get_leaf_rules(tree_model, feature_names)

# ------------------------------------------------
# 9) Summarize leaf-level uplift on validation data
# - Leafs are defined on train, but uplift evaluation is done on validation for honesty
# ------------------------------------------------
leaf_summary_valid = (
    valid_eval.groupby(["leaf_id", "treatment"])
              .agg(
                  n=("y", "count"),
                  purchase_rate=("y", "mean")
              )
              .reset_index()
)

leaf_pivot = leaf_summary_valid.pivot(index="leaf_id", columns="treatment", values=["n", "purchase_rate"])
leaf_pivot.columns = [
    "control_n" if col == ("n", 0) else
    "treated_n" if col == ("n", 1) else
    "control_purchase_rate" if col == ("purchase_rate", 0) else
    "treated_purchase_rate"
    for col in leaf_pivot.columns
]
leaf_pivot = leaf_pivot.reset_index()

for c in ["control_n", "treated_n", "control_purchase_rate", "treated_purchase_rate"]:
    if c not in leaf_pivot.columns:
        leaf_pivot[c] = np.nan

leaf_pivot["observed_uplift"] = leaf_pivot["treated_purchase_rate"] - leaf_pivot["control_purchase_rate"]

leaf_pred = (
    valid_eval.groupby("leaf_id")
              .agg(
                  valid_rows=("y", "count"),
                  avg_predicted_uplift=("uplift_score", "mean")
              )
              .reset_index()
)

leaf_report = leaf_pivot.merge(leaf_pred, on="leaf_id", how="left")
leaf_report["rule"] = leaf_report["leaf_id"].map(leaf_rule_map)

# Policy label
def label_policy(row):
    c_n = row.get("control_n", 0)
    t_n = row.get("treated_n", 0)
    u = row.get("observed_uplift", np.nan)

    if pd.isna(u):
        return "Insufficient_Data"
    if (pd.isna(c_n) or c_n < 500) or (pd.isna(t_n) or t_n < 500):
        return "Small_Sample"
    if u >= 0.020:
        return "Expand_B"
    elif u >= 0.010:
        return "Test_B"
    elif u >= 0.000:
        return "Neutral"
    else:
        return "Hold_Control"

leaf_report["policy_label"] = leaf_report.apply(label_policy, axis=1)

leaf_report = leaf_report.sort_values(
    ["policy_label", "observed_uplift", "valid_rows"],
    ascending=[True, False, False]
).reset_index(drop=True)

print("\n==================== Leaf-level Uplift Report (validation) ====================")
display(
    leaf_report[
        [
            "leaf_id", "valid_rows", "control_n", "treated_n",
            "control_purchase_rate", "treated_purchase_rate",
            "observed_uplift", "avg_predicted_uplift",
            "policy_label", "rule"
        ]
    ]
)

# ------------------------------------------------
# 10) Full scoring + top targets by leaf
# ------------------------------------------------
scored_tree = work_df[[
    "customer_id", "session_id", "experiment_group", "treatment", "y", "leaf_id"
]].copy()

for c in [
    "first_traffic_source", "first_device_type", "first_page_category",
    "session_campaign_id", "channel", "objective", "target_segment",
    "country", "age", "gender", "loyalty_tier", "acquisition_channel",
    "past_total_sessions", "past_total_purchases",
    "past_session_conversion_rate", "past_cart_abandon_sessions"
]:
    if c in work_df.columns:
        scored_tree[c] = work_df[c]

scored_tree["pred_uplift_tree"] = work_df["uplift_score_tree"]
scored_tree["leaf_rule"] = scored_tree["leaf_id"].map(leaf_rule_map)

# Top 20% by uplift
cutoff = scored_tree["pred_uplift_tree"].quantile(0.80)
scored_tree["target_flag_top20pct"] = (scored_tree["pred_uplift_tree"] >= cutoff).astype(int)

print("\n==================== Uplift Tree Example Top Targets ====================")
display(scored_tree.sort_values("pred_uplift_tree", ascending=False).head(20))

# ------------------------------------------------
# 11) Save outputs
# ------------------------------------------------
decile_path = os.path.join(BASE_PATH, "part4b_uplift_tree_decile_report.csv")
cum_path = os.path.join(BASE_PATH, "part4b_uplift_tree_cumulative_report.csv")
leaf_path = os.path.join(BASE_PATH, "part4b_uplift_tree_leaf_report.csv")
scored_path = os.path.join(BASE_PATH, "part4b_uplift_tree_scored_sessions.csv")
top_targets_path = os.path.join(BASE_PATH, "part4b_uplift_tree_top20_targets.csv")

decile_report_tree.to_csv(decile_path, index=False)
cum_report_tree.to_csv(cum_path, index=False)
leaf_report.to_csv(leaf_path, index=False)
scored_tree.to_csv(scored_path, index=False)
scored_tree[scored_tree["target_flag_top20pct"] == 1].to_csv(top_targets_path, index=False)

print("\n✅ PART 4B Uplift Tree complete")
print("Saved files:")
print("-", decile_path)
print("-", cum_path)
print("-", leaf_path)
print("-", scored_path)
print("-", top_targets_path)

# ------------------------------------------------
# 12) Interpretation summary
# ------------------------------------------------
best_leaf = leaf_report.sort_values("observed_uplift", ascending=False).head(1)
best_decile = decile_report_tree.sort_values("observed_uplift", ascending=False).head(1)

print("\n==================== Interpretation Summary ====================")
if len(best_leaf) == 1:
    r = best_leaf.iloc[0]
    print(
        f"[Best Leaf] leaf_id={int(r['leaf_id'])}, "
        f"observed uplift={r['observed_uplift']:.4f}, "
        f"policy={r['policy_label']}"
    )
    print(f"rule: {r['rule']}")
if len(best_decile) == 1:
    d = best_decile.iloc[0]
    print(
        f"[Best Decile] decile={int(d['uplift_decile'])}, "
        f"observed uplift={d['observed_uplift']:.4f}, "
        f"avg predicted uplift={d['avg_predicted_uplift']:.4f}"
    )

In [ ]:
# ================================================================
# PART 4B-2. Uplift Qini Curve / AUUC
# Assumptions:
# - valid_eval: validation dataframe created in PART 4B
#   columns: treatment, y, uplift_score
# - BASE_PATH
# Goal:
# - Quantify and visualize how much better the uplift model is than random targeting
# ================================================================
import os
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
from sklearn.metrics import auc

required_vars = ["valid_eval", "BASE_PATH"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise ValueError(f"Please run PART 4B first. Missing variables: {missing_vars}")

qini_df = valid_eval[["treatment", "y", "uplift_score"]].copy().dropna()
qini_df = qini_df.sort_values("uplift_score", ascending=False).reset_index(drop=True)

qini_df["cum_n"] = np.arange(1, len(qini_df) + 1)
qini_df["cum_treated_n"] = qini_df["treatment"].eq(1).cumsum()
qini_df["cum_control_n"] = qini_df["treatment"].eq(0).cumsum()

qini_df["treated_y"] = np.where(qini_df["treatment"] == 1, qini_df["y"], 0)
qini_df["control_y"] = np.where(qini_df["treatment"] == 0, qini_df["y"], 0)

qini_df["cum_treated_y"] = qini_df["treated_y"].cumsum()
qini_df["cum_control_y"] = qini_df["control_y"].cumsum()

# Qini curve:
# incremental gain = treated conversions - scaled control conversions
qini_df["cum_incremental_gain"] = np.where(
    qini_df["cum_control_n"] > 0,
    qini_df["cum_treated_y"] - qini_df["cum_control_y"] * (qini_df["cum_treated_n"] / qini_df["cum_control_n"]),
    np.nan
)

# cumulative uplift rate
qini_df["cum_treated_rate"] = np.where(
    qini_df["cum_treated_n"] > 0,
    qini_df["cum_treated_y"] / qini_df["cum_treated_n"],
    np.nan
)
qini_df["cum_control_rate"] = np.where(
    qini_df["cum_control_n"] > 0,
    qini_df["cum_control_y"] / qini_df["cum_control_n"],
    np.nan
)
qini_df["cum_uplift_rate"] = qini_df["cum_treated_rate"] - qini_df["cum_control_rate"]

qini_df["population_share"] = qini_df["cum_n"] / len(qini_df)

# Random-targeting baseline: connect the final incremental gain with a straight line
final_incremental_gain = qini_df["cum_incremental_gain"].dropna().iloc[-1]
qini_df["random_baseline_gain"] = qini_df["population_share"] * final_incremental_gain

# Overall uplift (used as an AUUC baseline reference)
overall_treated_rate = qini_df.loc[qini_df["treatment"] == 1, "y"].mean()
overall_control_rate = qini_df.loc[qini_df["treatment"] == 0, "y"].mean()
overall_uplift = overall_treated_rate - overall_control_rate

# Compute area under the curve
x = qini_df["population_share"].values
y_qini = np.nan_to_num(qini_df["cum_incremental_gain"].values, nan=0.0)
y_random = np.nan_to_num(qini_df["random_baseline_gain"].values, nan=0.0)
y_uplift = np.nan_to_num(qini_df["cum_uplift_rate"].values, nan=0.0)

qini_auc_model = auc(x, y_qini)
qini_auc_random = auc(x, y_random)
qini_gain = qini_auc_model - qini_auc_random

auuc_model = auc(x, y_uplift)
auuc_random = overall_uplift * 1.0   # area of the constant baseline over the [0, 1] interval
auuc_gain_vs_random = auuc_model - auuc_random

summary_qini = pd.DataFrame({
    "metric": ["final_incremental_gain", "overall_uplift", "qini_auc_model", "qini_auc_random", "qini_gain", "auuc_model", "auuc_random", "auuc_gain_vs_random"],
    "value": [final_incremental_gain, overall_uplift, qini_auc_model, qini_auc_random, qini_gain, auuc_model, auuc_random, auuc_gain_vs_random]
})

print("\n==================== Uplift Qini / AUUC Summary ====================")
display(summary_qini)

# plot
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(qini_df["population_share"], qini_df["cum_incremental_gain"], label="Model Qini Curve")
plt.plot(qini_df["population_share"], qini_df["random_baseline_gain"], linestyle="--", label="Random Targeting")
plt.xlabel("Population Share Targeted")
plt.ylabel("Cumulative Incremental Gain")
plt.title("Qini Curve")
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(qini_df["population_share"], qini_df["cum_uplift_rate"], label="Model Cumulative Uplift Rate")
plt.axhline(overall_uplift, linestyle="--", label="Overall Uplift (Random Baseline)")
plt.xlabel("Population Share Targeted")
plt.ylabel("Cumulative Uplift Rate")
plt.title("AUUC-style Uplift Curve")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Save outputs
qini_curve_path = os.path.join(BASE_PATH, "part4b_uplift_qini_curve_data.csv")
qini_summary_path = os.path.join(BASE_PATH, "part4b_uplift_qini_auuc_summary.csv")

qini_df.to_csv(qini_curve_path, index=False)
summary_qini.to_csv(qini_summary_path, index=False)

print("\n✅ PART 4B-2 complete")
print("-", qini_curve_path)
print("-", qini_summary_path)

print("\n==================== Interpretation Summary ====================")
print(f"Final incremental gain: {final_incremental_gain:.4f}")
print(f"Overall uplift: {overall_uplift:.4f}")
print(f"Qini gain vs random: {qini_gain:.4f}")
print(f"AUUC gain vs random baseline: {auuc_gain_vs_random:.6f}")

In [ ]:
# ================================================================
# PART 4B-3. Bootstrap CI for uplift
# Goal:
# - Compute CIs for overall uplift, top-20% uplift, and cart top-decile rate
# - Here we calculate the uplift-related CIs first
# ================================================================
import os
import numpy as np
import pandas as pd
from IPython.display import display

required_vars = ["valid_eval", "BASE_PATH"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise ValueError(f"Please run PART 4B first. Missing variables: {missing_vars}")

BOOT_N = 300   # Use 200 for speed, 500 for more stability
RANDOM_STATE = 42
TOP_SHARE = 0.20

boot_df = valid_eval[["treatment", "y", "uplift_score"]].copy().dropna().reset_index(drop=True)

y_arr = boot_df["y"].to_numpy()
t_arr = boot_df["treatment"].to_numpy()
s_arr = boot_df["uplift_score"].to_numpy()
n = len(boot_df)

def safe_uplift(y, t):
    tr = (t == 1)
    ct = (t == 0)
    if tr.sum() == 0 or ct.sum() == 0:
        return np.nan
    return y[tr].mean() - y[ct].mean()

def top_share_uplift(score, y, t, top_share=0.20):
    n_local = len(score)
    k = max(1, int(n_local * top_share))
    idx_top = np.argpartition(-score, k - 1)[:k]
    return safe_uplift(y[idx_top], t[idx_top])

point_overall_uplift = safe_uplift(y_arr, t_arr)
point_top20_uplift = top_share_uplift(s_arr, y_arr, t_arr, TOP_SHARE)

rng = np.random.default_rng(RANDOM_STATE)
boot_rows = []

for b in range(BOOT_N):
    idx = rng.integers(0, n, size=n)
    y_b = y_arr[idx]
    t_b = t_arr[idx]
    s_b = s_arr[idx]

    boot_rows.append({
        "bootstrap_iter": b + 1,
        "overall_uplift": safe_uplift(y_b, t_b),
        "top20_uplift": top_share_uplift(s_b, y_b, t_b, TOP_SHARE),
    })

boot_res = pd.DataFrame(boot_rows)

def ci_summary(series, point_estimate, metric_name):
    series = pd.to_numeric(series, errors="coerce").dropna()
    return {
        "metric": metric_name,
        "point_estimate": point_estimate,
        "bootstrap_mean": series.mean(),
        "ci_2.5": series.quantile(0.025),
        "ci_97.5": series.quantile(0.975),
        "boot_n_used": len(series)
    }

ci_table = pd.DataFrame([
    ci_summary(boot_res["overall_uplift"], point_overall_uplift, "overall_uplift"),
    ci_summary(boot_res["top20_uplift"], point_top20_uplift, "top20_uplift"),
])

print("\n==================== Uplift Bootstrap CI ====================")
display(ci_table)

# Save outputs
boot_detail_path = os.path.join(BASE_PATH, "part4b_uplift_bootstrap_detail.csv")
boot_ci_path = os.path.join(BASE_PATH, "part4b_uplift_bootstrap_ci_summary.csv")

boot_res.to_csv(boot_detail_path, index=False)
ci_table.to_csv(boot_ci_path, index=False)

print("\n✅ PART 4B-3 complete")
print("-", boot_detail_path)
print("-", boot_ci_path)

print("\n==================== Interpretation Summary ====================")
for _, r in ci_table.iterrows():
    print(
        f"{r['metric']}: point={r['point_estimate']:.4f}, "
        f"95% CI=[{r['ci_2.5']:.4f}, {r['ci_97.5']:.4f}]"
    )

In [ ]:
# ================================================================
# PART 4B-4. Campaign Incrementality Analysis
# Goal:
# - Compute actual uplift by session_campaign_id
# - Compare expected_uplift vs actual uplift
# - Analyze gaps by channel / objective / target_segment
# Assumptions:
# - df and BASE_PATH must exist
# - df must contain experiment_group, any_purchase (or y), expected_uplift,
#   session_campaign_id, channel, objective, and target_segment
# ================================================================
import os
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt

# ------------------------------------------------
# 0) Check assumptions
# ------------------------------------------------
required_vars = ["df", "BASE_PATH"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise ValueError(f"Please prepare df and BASE_PATH first. Missing variables: {missing_vars}")

work_df = df.copy()

# Prepare treatment / outcome
if "treatment" not in work_df.columns:
    if "experiment_group" not in work_df.columns:
        raise ValueError("experiment_group or treatment column is required.")
    work_df["treatment"] = np.where(work_df["experiment_group"] == "Variant_B", 1, 0)

if "y" not in work_df.columns:
    if "any_purchase" in work_df.columns:
        work_df["y"] = pd.to_numeric(work_df["any_purchase"], errors="coerce").fillna(0).astype(int)
    else:
        raise ValueError("y or any_purchase column is required.")

# Required columns for analysis
needed_cols = [
    "treatment", "y", "expected_uplift",
    "session_campaign_id", "channel", "objective", "target_segment"
]
missing_cols = [c for c in needed_cols if c not in work_df.columns]
if missing_cols:
    raise ValueError(f"df is missing required columns: {missing_cols}")

# Remove anomalous experiment_group values such as Mixed
if "experiment_group" in work_df.columns:
    work_df = work_df[work_df["experiment_group"].isin(["Control", "Variant_B"])].copy()

# Clean dtypes
work_df["expected_uplift"] = pd.to_numeric(work_df["expected_uplift"], errors="coerce")
work_df["session_campaign_id"] = pd.to_numeric(work_df["session_campaign_id"], errors="coerce").fillna(0).astype(int)

for c in ["channel", "objective", "target_segment"]:
    work_df[c] = work_df[c].astype("string").fillna("Unknown")

# ------------------------------------------------
# 1) Helper: compute actual uplift by group
# ------------------------------------------------
def uplift_by_group(df_in, group_col, min_rows=500, min_treat=100, min_control=100):
    tmp = df_in[[group_col, "treatment", "y", "expected_uplift"]].copy()

    out = (
        tmp.groupby(group_col)
           .apply(lambda x: pd.Series({
               "n": len(x),
               "treated_n": int((x["treatment"] == 1).sum()),
               "control_n": int((x["treatment"] == 0).sum()),
               "treated_purchase_rate": x.loc[x["treatment"] == 1, "y"].mean() if (x["treatment"] == 1).sum() > 0 else np.nan,
               "control_purchase_rate": x.loc[x["treatment"] == 0, "y"].mean() if (x["treatment"] == 0).sum() > 0 else np.nan,
               "actual_uplift": (
                   x.loc[x["treatment"] == 1, "y"].mean() - x.loc[x["treatment"] == 0, "y"].mean()
               ) if ((x["treatment"] == 1).sum() > 0 and (x["treatment"] == 0).sum() > 0) else np.nan,
               "expected_uplift_mean": x["expected_uplift"].mean(),
               "expected_uplift_median": x["expected_uplift"].median()
           }))
           .reset_index()
    )

    out["gap_actual_minus_expected"] = out["actual_uplift"] - out["expected_uplift_mean"]
    out["abs_gap"] = out["gap_actual_minus_expected"].abs()

    def label_gap(row):
        if pd.isna(row["actual_uplift"]) or pd.isna(row["expected_uplift_mean"]):
            return "Insufficient_Data"
        if row["n"] < min_rows or row["treated_n"] < min_treat or row["control_n"] < min_control:
            return "Small_Sample"
        gap = row["gap_actual_minus_expected"]
        if gap >= 0.01:
            return "Overperform"
        elif gap <= -0.01:
            return "Underperform"
        else:
            return "On_Expectation"

    out["performance_label"] = out.apply(label_gap, axis=1)

    out["is_stable_group"] = (
        (out["n"] >= min_rows) &
        (out["treated_n"] >= min_treat) &
        (out["control_n"] >= min_control)
    ).astype(int)

    return out.sort_values(["is_stable_group", "actual_uplift", "n"], ascending=[False, False, False]).reset_index(drop=True)

# ------------------------------------------------
# 2) Summarize the overall average treatment effect
# ------------------------------------------------
overall_summary = pd.DataFrame([{
    "n": len(work_df),
    "treated_n": int((work_df["treatment"] == 1).sum()),
    "control_n": int((work_df["treatment"] == 0).sum()),
    "treated_purchase_rate": work_df.loc[work_df["treatment"] == 1, "y"].mean(),
    "control_purchase_rate": work_df.loc[work_df["treatment"] == 0, "y"].mean(),
    "actual_uplift": work_df.loc[work_df["treatment"] == 1, "y"].mean() - work_df.loc[work_df["treatment"] == 0, "y"].mean(),
    "expected_uplift_mean": work_df["expected_uplift"].mean()
}])

overall_summary["gap_actual_minus_expected"] = overall_summary["actual_uplift"] - overall_summary["expected_uplift_mean"]

print("\n==================== Overall Incrementality Summary ====================")
display(overall_summary)

# ------------------------------------------------
# 3) Group-level incrementality table
# ------------------------------------------------
campaign_report = uplift_by_group(work_df, "session_campaign_id", min_rows=500, min_treat=100, min_control=100)
channel_report = uplift_by_group(work_df, "channel", min_rows=1000, min_treat=200, min_control=200)
objective_report = uplift_by_group(work_df, "objective", min_rows=1000, min_treat=200, min_control=200)
segment_report = uplift_by_group(work_df, "target_segment", min_rows=1000, min_treat=200, min_control=200)

print("\n==================== Campaign Incrementality: session_campaign_id ====================")
display(campaign_report.head(20))

print("\n==================== Channel Incrementality ====================")
display(channel_report)

print("\n==================== Objective Incrementality ====================")
display(objective_report)

print("\n==================== Target Segment Incrementality ====================")
display(segment_report)

# ------------------------------------------------
# 4) Summary statistics for expected vs actual
# ------------------------------------------------
stable_campaigns = campaign_report[campaign_report["is_stable_group"] == 1].copy()

if len(stable_campaigns) > 1:
    corr_expected_actual = stable_campaigns["expected_uplift_mean"].corr(stable_campaigns["actual_uplift"])
    weighted_mae = np.average(
        np.abs(stable_campaigns["actual_uplift"] - stable_campaigns["expected_uplift_mean"]),
        weights=stable_campaigns["n"]
    )
else:
    corr_expected_actual = np.nan
    weighted_mae = np.nan

expected_actual_summary = pd.DataFrame([{
    "stable_campaign_count": len(stable_campaigns),
    "corr_expected_vs_actual": corr_expected_actual,
    "weighted_mae_actual_vs_expected": weighted_mae
}])

print("\n==================== Expected vs Actual Summary ====================")
display(expected_actual_summary)

# ------------------------------------------------
# 5) Example top / bottom campaigns
# ------------------------------------------------
top_campaigns = stable_campaigns.sort_values("actual_uplift", ascending=False).head(10)
bottom_campaigns = stable_campaigns.sort_values("actual_uplift", ascending=True).head(10)
most_overperform = stable_campaigns.sort_values("gap_actual_minus_expected", ascending=False).head(10)
most_underperform = stable_campaigns.sort_values("gap_actual_minus_expected", ascending=True).head(10)

print("\n==================== Top 10 Actual Uplift Campaigns ====================")
display(top_campaigns)

print("\n==================== Bottom 10 Actual Uplift Campaigns ====================")
display(bottom_campaigns)

print("\n==================== Most Overperforming Campaigns (Actual - Expected) ====================")
display(most_overperform)

print("\n==================== Most Underperforming Campaigns (Actual - Expected) ====================")
display(most_underperform)

# ------------------------------------------------
# 6) Visualization
# ------------------------------------------------
# 6-1) expected vs actual scatter (campaign)
if len(stable_campaigns) > 0:
    plt.figure(figsize=(7, 6))
    plt.scatter(
        stable_campaigns["expected_uplift_mean"],
        stable_campaigns["actual_uplift"],
        s=np.clip(stable_campaigns["n"] / 50, 20, 300),
        alpha=0.7
    )
    lim_min = min(stable_campaigns["expected_uplift_mean"].min(), stable_campaigns["actual_uplift"].min())
    lim_max = max(stable_campaigns["expected_uplift_mean"].max(), stable_campaigns["actual_uplift"].max())
    plt.plot([lim_min, lim_max], [lim_min, lim_max], linestyle="--")
    plt.xlabel("Expected Uplift (mean)")
    plt.ylabel("Actual Uplift")
    plt.title("Campaign Expected vs Actual Uplift")
    plt.grid(alpha=0.3)
    plt.show()

# 6-2) channel gap bar
channel_plot = channel_report.sort_values("gap_actual_minus_expected", ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(channel_plot["channel"].astype(str), channel_plot["gap_actual_minus_expected"])
plt.title("Channel-level Gap: Actual Uplift - Expected Uplift")
plt.xlabel("Channel")
plt.ylabel("Gap")
plt.xticks(rotation=30)
plt.show()

# 6-3) objective gap bar
objective_plot = objective_report.sort_values("gap_actual_minus_expected", ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(objective_plot["objective"].astype(str), objective_plot["gap_actual_minus_expected"])
plt.title("Objective-level Gap: Actual Uplift - Expected Uplift")
plt.xlabel("Objective")
plt.ylabel("Gap")
plt.xticks(rotation=30)
plt.show()

# 6-4) segment gap bar
segment_plot = segment_report.sort_values("gap_actual_minus_expected", ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(segment_plot["target_segment"].astype(str), segment_plot["gap_actual_minus_expected"])
plt.title("Target Segment-level Gap: Actual Uplift - Expected Uplift")
plt.xlabel("Target Segment")
plt.ylabel("Gap")
plt.xticks(rotation=30)
plt.show()

# ------------------------------------------------
# 7) Save outputs
# ------------------------------------------------
os.makedirs(BASE_PATH, exist_ok=True)

overall_path = os.path.join(BASE_PATH, "part4b_campaign_incrementality_overall_summary.csv")
campaign_path = os.path.join(BASE_PATH, "part4b_campaign_incrementality_by_campaign.csv")
channel_path = os.path.join(BASE_PATH, "part4b_campaign_incrementality_by_channel.csv")
objective_path = os.path.join(BASE_PATH, "part4b_campaign_incrementality_by_objective.csv")
segment_path = os.path.join(BASE_PATH, "part4b_campaign_incrementality_by_target_segment.csv")
expected_actual_path = os.path.join(BASE_PATH, "part4b_campaign_expected_vs_actual_summary.csv")

overall_summary.to_csv(overall_path, index=False)
campaign_report.to_csv(campaign_path, index=False)
channel_report.to_csv(channel_path, index=False)
objective_report.to_csv(objective_path, index=False)
segment_report.to_csv(segment_path, index=False)
expected_actual_summary.to_csv(expected_actual_path, index=False)

print("\n✅ PART 4B-4 Campaign Incrementality complete")
print("-", overall_path)
print("-", campaign_path)
print("-", channel_path)
print("-", objective_path)
print("-", segment_path)
print("-", expected_actual_path)

# ------------------------------------------------
# 8) Interpretation Summary
# ------------------------------------------------
print("\n==================== Interpretation Summary ====================")

overall_actual = overall_summary.loc[0, "actual_uplift"]
overall_expected = overall_summary.loc[0, "expected_uplift_mean"]
overall_gap = overall_summary.loc[0, "gap_actual_minus_expected"]

print(f"Overall actual uplift: {overall_actual:.4f}")
print(f"Mean expected uplift overall: {overall_expected:.4f}")
print(f"Overall gap (actual - expected): {overall_gap:.4f}")

if len(stable_campaigns) > 0:
    best_campaign = stable_campaigns.sort_values("actual_uplift", ascending=False).iloc[0]
    worst_campaign = stable_campaigns.sort_values("actual_uplift", ascending=True).iloc[0]
    print(
        f"Most stable high-uplift campaign: campaign_id={int(best_campaign['session_campaign_id'])}, "
        f"actual={best_campaign['actual_uplift']:.4f}, expected={best_campaign['expected_uplift_mean']:.4f}, "
        f"gap={best_campaign['gap_actual_minus_expected']:.4f}"
    )
    print(
        f"Most stable low-uplift campaign: campaign_id={int(worst_campaign['session_campaign_id'])}, "
        f"actual={worst_campaign['actual_uplift']:.4f}, expected={worst_campaign['expected_uplift_mean']:.4f}, "
        f"gap={worst_campaign['gap_actual_minus_expected']:.4f}"
    )

if pd.notna(corr_expected_actual):
    print(f"Expected vs actual correlation for stable campaigns: {corr_expected_actual:.4f}")
if pd.notna(weighted_mae):
    print(f"Weighted MAE for expected vs actual on stable campaigns: {weighted_mae:.4f}")

In [ ]:
# ================================================================
# PART 4B-4R. Clean Campaign Incrementality Re-analysis
# - Exclude Unknown and campaign_id=0
# - Cleaner incrementality table for practical interpretation
# Assumptions:
#   df and BASE_PATH must exist
# ================================================================
import os
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt

required_vars = ["df", "BASE_PATH"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise ValueError(f"Please prepare df and BASE_PATH first. Missing variables: {missing_vars}")

work_df = df.copy()

# ------------------------------------------------
# 0) Prepare treatment / outcome
# ------------------------------------------------
if "treatment" not in work_df.columns:
    if "experiment_group" not in work_df.columns:
        raise ValueError("experiment_group or treatment column is required.")
    work_df["treatment"] = np.where(work_df["experiment_group"] == "Variant_B", 1, 0)

if "y" not in work_df.columns:
    if "any_purchase" in work_df.columns:
        work_df["y"] = pd.to_numeric(work_df["any_purchase"], errors="coerce").fillna(0).astype(int)
    else:
        raise ValueError("y or any_purchase column is required.")

need_cols = ["session_campaign_id", "channel", "objective", "target_segment", "expected_uplift"]
missing_cols = [c for c in need_cols if c not in work_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

work_df["session_campaign_id"] = pd.to_numeric(work_df["session_campaign_id"], errors="coerce").fillna(0).astype(int)
work_df["expected_uplift"] = pd.to_numeric(work_df["expected_uplift"], errors="coerce")
for c in ["channel", "objective", "target_segment"]:
    work_df[c] = work_df[c].astype("string").fillna("Unknown")

if "experiment_group" in work_df.columns:
    work_df = work_df[work_df["experiment_group"].isin(["Control", "Variant_B"])].copy()

# ------------------------------------------------
# 1) Exclude Unknown / 0
# ------------------------------------------------
clean_df = work_df[
    (work_df["session_campaign_id"] != 0) &
    (work_df["channel"] != "Unknown") &
    (work_df["objective"] != "Unknown") &
    (work_df["target_segment"] != "Unknown")
].copy()

print("✅ clean_df rows:", f"{len(clean_df):,}")
print("✅ campaign_id unique:", clean_df["session_campaign_id"].nunique())
print("✅ channels:", clean_df["channel"].nunique())
print("✅ objectives:", clean_df["objective"].nunique())
print("✅ target_segments:", clean_df["target_segment"].nunique())

# ------------------------------------------------
# 2) helper
# ------------------------------------------------
def uplift_by_group(df_in, group_col, min_rows=500, min_treat=100, min_control=100):
    tmp = df_in[[group_col, "treatment", "y", "expected_uplift"]].copy()

    out = (
        tmp.groupby(group_col)
           .apply(lambda x: pd.Series({
               "n": len(x),
               "treated_n": int((x["treatment"] == 1).sum()),
               "control_n": int((x["treatment"] == 0).sum()),
               "treated_purchase_rate": x.loc[x["treatment"] == 1, "y"].mean() if (x["treatment"] == 1).sum() > 0 else np.nan,
               "control_purchase_rate": x.loc[x["treatment"] == 0, "y"].mean() if (x["treatment"] == 0).sum() > 0 else np.nan,
               "actual_uplift": (
                   x.loc[x["treatment"] == 1, "y"].mean() - x.loc[x["treatment"] == 0, "y"].mean()
               ) if ((x["treatment"] == 1).sum() > 0 and (x["treatment"] == 0).sum() > 0) else np.nan,
               "expected_uplift_mean": x["expected_uplift"].mean(),
               "expected_uplift_median": x["expected_uplift"].median()
           }))
           .reset_index()
    )

    out["gap_actual_minus_expected"] = out["actual_uplift"] - out["expected_uplift_mean"]
    out["abs_gap"] = out["gap_actual_minus_expected"].abs()
    out["is_stable_group"] = (
        (out["n"] >= min_rows) &
        (out["treated_n"] >= min_treat) &
        (out["control_n"] >= min_control)
    ).astype(int)

    def label_perf(row):
        if pd.isna(row["actual_uplift"]) or pd.isna(row["expected_uplift_mean"]):
            return "Insufficient_Data"
        if row["is_stable_group"] == 0:
            return "Small_Sample"
        gap = row["gap_actual_minus_expected"]
        if gap >= 0.01:
            return "Overperform"
        elif gap <= -0.01:
            return "Underperform"
        else:
            return "On_Expectation"

    out["performance_label"] = out.apply(label_perf, axis=1)

    return out.sort_values(
        ["is_stable_group", "actual_uplift", "n"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

# ------------------------------------------------
# 3) Re-run analysis overall and by group
# ------------------------------------------------
overall_clean = pd.DataFrame([{
    "n": len(clean_df),
    "treated_n": int((clean_df["treatment"] == 1).sum()),
    "control_n": int((clean_df["treatment"] == 0).sum()),
    "treated_purchase_rate": clean_df.loc[clean_df["treatment"] == 1, "y"].mean(),
    "control_purchase_rate": clean_df.loc[clean_df["treatment"] == 0, "y"].mean(),
    "actual_uplift": clean_df.loc[clean_df["treatment"] == 1, "y"].mean() - clean_df.loc[clean_df["treatment"] == 0, "y"].mean(),
    "expected_uplift_mean": clean_df["expected_uplift"].mean()
}])
overall_clean["gap_actual_minus_expected"] = overall_clean["actual_uplift"] - overall_clean["expected_uplift_mean"]

campaign_clean = uplift_by_group(clean_df, "session_campaign_id", min_rows=500, min_treat=100, min_control=100)
channel_clean = uplift_by_group(clean_df, "channel", min_rows=1000, min_treat=200, min_control=200)
objective_clean = uplift_by_group(clean_df, "objective", min_rows=1000, min_treat=200, min_control=200)
segment_clean = uplift_by_group(clean_df, "target_segment", min_rows=1000, min_treat=200, min_control=200)

print("\n==================== Clean Overall Incrementality ====================")
display(overall_clean)

print("\n==================== Clean Campaign Incrementality ====================")
display(campaign_clean.head(20))

print("\n==================== Clean Channel Incrementality ====================")
display(channel_clean)

print("\n==================== Clean Objective Incrementality ====================")
display(objective_clean)

print("\n==================== Clean Target Segment Incrementality ====================")
display(segment_clean)

# ------------------------------------------------
# 4) expected vs actual summary (clean)
# ------------------------------------------------
stable_campaign_clean = campaign_clean[campaign_clean["is_stable_group"] == 1].copy()

if len(stable_campaign_clean) > 1:
    corr_clean = stable_campaign_clean["expected_uplift_mean"].corr(stable_campaign_clean["actual_uplift"])
    weighted_mae_clean = np.average(
        np.abs(stable_campaign_clean["actual_uplift"] - stable_campaign_clean["expected_uplift_mean"]),
        weights=stable_campaign_clean["n"]
    )
else:
    corr_clean = np.nan
    weighted_mae_clean = np.nan

expected_actual_clean = pd.DataFrame([{
    "stable_campaign_count": len(stable_campaign_clean),
    "corr_expected_vs_actual": corr_clean,
    "weighted_mae_actual_vs_expected": weighted_mae_clean
}])

print("\n==================== Clean Expected vs Actual Summary ====================")
display(expected_actual_clean)

# ------------------------------------------------
# 5) Visualization
# ------------------------------------------------
if len(stable_campaign_clean) > 0:
    plt.figure(figsize=(7, 6))
    plt.scatter(
        stable_campaign_clean["expected_uplift_mean"],
        stable_campaign_clean["actual_uplift"],
        s=np.clip(stable_campaign_clean["n"] / 50, 20, 300),
        alpha=0.7
    )
    lim_min = min(stable_campaign_clean["expected_uplift_mean"].min(), stable_campaign_clean["actual_uplift"].min())
    lim_max = max(stable_campaign_clean["expected_uplift_mean"].max(), stable_campaign_clean["actual_uplift"].max())
    plt.plot([lim_min, lim_max], [lim_min, lim_max], linestyle="--")
    plt.xlabel("Expected Uplift (mean)")
    plt.ylabel("Actual Uplift")
    plt.title("Clean Campaign Expected vs Actual Uplift")
    plt.grid(alpha=0.3)
    plt.show()

for title, plot_df, xcol in [
    ("Clean Channel Gap: Actual - Expected", channel_clean.sort_values("gap_actual_minus_expected", ascending=False), "channel"),
    ("Clean Objective Gap: Actual - Expected", objective_clean.sort_values("gap_actual_minus_expected", ascending=False), "objective"),
    ("Clean Target Segment Gap: Actual - Expected", segment_clean.sort_values("gap_actual_minus_expected", ascending=False), "target_segment")
]:
    plt.figure(figsize=(8, 5))
    plt.bar(plot_df[xcol].astype(str), plot_df["gap_actual_minus_expected"])
    plt.title(title)
    plt.xlabel(xcol)
    plt.ylabel("Gap")
    plt.xticks(rotation=30)
    plt.show()

# ------------------------------------------------
# 6) Save outputs
# ------------------------------------------------
os.makedirs(BASE_PATH, exist_ok=True)

overall_clean_path = os.path.join(BASE_PATH, "part4b_campaign_incrementality_overall_summary_clean.csv")
campaign_clean_path = os.path.join(BASE_PATH, "part4b_campaign_incrementality_by_campaign_clean.csv")
channel_clean_path = os.path.join(BASE_PATH, "part4b_campaign_incrementality_by_channel_clean.csv")
objective_clean_path = os.path.join(BASE_PATH, "part4b_campaign_incrementality_by_objective_clean.csv")
segment_clean_path = os.path.join(BASE_PATH, "part4b_campaign_incrementality_by_target_segment_clean.csv")
expected_actual_clean_path = os.path.join(BASE_PATH, "part4b_campaign_expected_vs_actual_summary_clean.csv")

overall_clean.to_csv(overall_clean_path, index=False)
campaign_clean.to_csv(campaign_clean_path, index=False)
channel_clean.to_csv(channel_clean_path, index=False)
objective_clean.to_csv(objective_clean_path, index=False)
segment_clean.to_csv(segment_clean_path, index=False)
expected_actual_clean.to_csv(expected_actual_clean_path, index=False)

print("\n✅ PART 4B-4R complete")
print("-", overall_clean_path)
print("-", campaign_clean_path)
print("-", channel_clean_path)
print("-", objective_clean_path)
print("-", segment_clean_path)
print("-", expected_actual_clean_path)

In [ ]:
# ================================================================
# PART 4B-5. first_page_category × Segment × Channel Actual Uplift Strategy Table
# Goal:
# - Identify which first_page_category × target_segment × channel combinations
#   have high actual uplift and build a strategy table
# Assumptions:
#   df and BASE_PATH must exist
# ================================================================
import os
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt

# ------------------------------------------------
# 0) Check assumptions
# ------------------------------------------------
required_vars = ["df", "BASE_PATH"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise ValueError(f"Please prepare df and BASE_PATH first. Missing variables: {missing_vars}")

work_df = df.copy()

# ------------------------------------------------
# 1) Prepare treatment / outcome
# ------------------------------------------------
if "treatment" not in work_df.columns:
    if "experiment_group" not in work_df.columns:
        raise ValueError("experiment_group or treatment column is required.")
    work_df["treatment"] = np.where(work_df["experiment_group"] == "Variant_B", 1, 0)

if "y" not in work_df.columns:
    if "any_purchase" in work_df.columns:
        work_df["y"] = pd.to_numeric(work_df["any_purchase"], errors="coerce").fillna(0).astype(int)
    else:
        raise ValueError("y or any_purchase column is required.")

# ------------------------------------------------
# 2) Select the product-like column explicitly
# ------------------------------------------------
product_dim = "first_page_category"

required_cols = [
    product_dim,
    "channel",
    "target_segment",
    "expected_uplift",
    "session_campaign_id",
    "treatment",
    "y"
]
missing_cols = [c for c in required_cols if c not in work_df.columns]
if missing_cols:
    raise ValueError(f"df is missing required columns: {missing_cols}")

print(f"✅ Product-like analysis column used: {product_dim}")

# ------------------------------------------------
# 3) Base filtering
#    - Use only Control / Variant_B
#    - Remove campaign_id=0
#    - Remove Unknown
# ------------------------------------------------
if "experiment_group" in work_df.columns:
    work_df = work_df[work_df["experiment_group"].isin(["Control", "Variant_B"])].copy()

work_df["session_campaign_id"] = pd.to_numeric(
    work_df["session_campaign_id"], errors="coerce"
).fillna(0).astype(int)

work_df["expected_uplift"] = pd.to_numeric(
    work_df["expected_uplift"], errors="coerce"
)

for c in [product_dim, "channel", "target_segment"]:
    work_df[c] = work_df[c].astype("string").fillna("Unknown")

combo_df = work_df[
    (work_df["session_campaign_id"] != 0) &
    (work_df["channel"] != "Unknown") &
    (work_df["target_segment"] != "Unknown") &
    (work_df[product_dim] != "Unknown")
].copy()

print("✅ combo_df rows:", f"{len(combo_df):,}")
print("✅ unique first_page_category:", combo_df[product_dim].nunique())
print("✅ unique target_segment:", combo_df["target_segment"].nunique())
print("✅ unique channel:", combo_df["channel"].nunique())

# ------------------------------------------------
# 4) helper
# ------------------------------------------------
def uplift_by_combo(df_in, group_cols, min_rows=400, min_treat=80, min_control=80):
    tmp = df_in[group_cols + ["treatment", "y", "expected_uplift"]].copy()

    out = (
        tmp.groupby(group_cols)
           .apply(lambda x: pd.Series({
               "n": len(x),
               "treated_n": int((x["treatment"] == 1).sum()),
               "control_n": int((x["treatment"] == 0).sum()),
               "treated_purchase_rate": (
                   x.loc[x["treatment"] == 1, "y"].mean()
                   if (x["treatment"] == 1).sum() > 0 else np.nan
               ),
               "control_purchase_rate": (
                   x.loc[x["treatment"] == 0, "y"].mean()
                   if (x["treatment"] == 0).sum() > 0 else np.nan
               ),
               "actual_uplift": (
                   x.loc[x["treatment"] == 1, "y"].mean() -
                   x.loc[x["treatment"] == 0, "y"].mean()
               ) if ((x["treatment"] == 1).sum() > 0 and (x["treatment"] == 0).sum() > 0) else np.nan,
               "expected_uplift_mean": x["expected_uplift"].mean(),
               "expected_uplift_median": x["expected_uplift"].median()
           }))
           .reset_index()
    )

    out["gap_actual_minus_expected"] = out["actual_uplift"] - out["expected_uplift_mean"]
    out["abs_gap"] = out["gap_actual_minus_expected"].abs()

    out["is_stable_group"] = (
        (out["n"] >= min_rows) &
        (out["treated_n"] >= min_treat) &
        (out["control_n"] >= min_control)
    ).astype(int)

    def action_label(row):
        if pd.isna(row["actual_uplift"]):
            return "Insufficient_Data"
        if row["is_stable_group"] == 0:
            return "Small_Sample"

        u = row["actual_uplift"]
        if u >= 0.020:
            return "Priority_Expand"
        elif u >= 0.010:
            return "Test_or_Scale"
        elif u >= 0.000:
            return "Neutral"
        else:
            return "Avoid_or_Rework"

    out["action_label"] = out.apply(action_label, axis=1)

    return out.sort_values(
        ["is_stable_group", "actual_uplift", "n"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

# ------------------------------------------------
# 5) 3-way combo table
# ------------------------------------------------
group_cols = [product_dim, "target_segment", "channel"]

combo_report = uplift_by_combo(
    combo_df,
    group_cols=group_cols,
    min_rows=400,
    min_treat=80,
    min_control=80
)

print("\n==================== first_page_category × Segment × Channel Actual Uplift ====================")
display(combo_report.head(30))

stable_combo = combo_report[combo_report["is_stable_group"] == 1].copy()

top_expand = stable_combo.sort_values("actual_uplift", ascending=False).head(20)
bottom_combo = stable_combo.sort_values("actual_uplift", ascending=True).head(20)
best_gap = stable_combo.sort_values("gap_actual_minus_expected", ascending=False).head(20)
worst_gap = stable_combo.sort_values("gap_actual_minus_expected", ascending=True).head(20)

print("\n==================== Top 20 Priority Combos ====================")
display(top_expand)

print("\n==================== Bottom 20 Combos ====================")
display(bottom_combo)

print("\n==================== Most Overperforming Combos (Actual - Expected) ====================")
display(best_gap)

print("\n==================== Most Underperforming Combos (Actual - Expected) ====================")
display(worst_gap)

# ------------------------------------------------
# 6) Strategy summary table
# ------------------------------------------------
strategy_table = stable_combo[
    group_cols + [
        "n",
        "treated_n",
        "control_n",
        "treated_purchase_rate",
        "control_purchase_rate",
        "actual_uplift",
        "expected_uplift_mean",
        "gap_actual_minus_expected",
        "action_label"
    ]
].copy()

action_order = {
    "Priority_Expand": 0,
    "Test_or_Scale": 1,
    "Neutral": 2,
    "Avoid_or_Rework": 3,
    "Small_Sample": 4,
    "Insufficient_Data": 5
}
strategy_table["action_order"] = strategy_table["action_label"].map(action_order).fillna(99)

strategy_table = strategy_table.sort_values(
    ["action_order", "actual_uplift", "n"],
    ascending=[True, False, False]
).drop(columns=["action_order"]).reset_index(drop=True)

print("\n==================== Strategy Table ====================")
display(strategy_table.head(50))

# ------------------------------------------------
# 7) Weighted uplift pivot: channel × segment
# ------------------------------------------------
pivot_channel_segment = (
    stable_combo.groupby(["channel", "target_segment"], as_index=False)
               .apply(lambda x: pd.Series({
                   "weighted_actual_uplift": np.average(x["actual_uplift"], weights=x["n"]),
                   "total_n": x["n"].sum()
               }))
               .reset_index(drop=True)
)

pivot_mat = pivot_channel_segment.pivot(
    index="target_segment",
    columns="channel",
    values="weighted_actual_uplift"
)

print("\n==================== Channel × Segment Weighted Actual Uplift Pivot ====================")
display(pivot_mat)

if pivot_mat.shape[0] > 0 and pivot_mat.shape[1] > 0:
    plt.figure(figsize=(8, 5))
    plt.imshow(pivot_mat.fillna(0).values, aspect="auto")
    plt.colorbar(label="Weighted Actual Uplift")
    plt.xticks(range(len(pivot_mat.columns)), pivot_mat.columns, rotation=30)
    plt.yticks(range(len(pivot_mat.index)), pivot_mat.index)
    plt.title("Channel × Segment Weighted Actual Uplift")
    plt.show()

# ------------------------------------------------
# 8) first_page_category × channel weighted uplift pivot
# ------------------------------------------------
pivot_category_channel = (
    stable_combo.groupby([product_dim, "channel"], as_index=False)
               .apply(lambda x: pd.Series({
                   "weighted_actual_uplift": np.average(x["actual_uplift"], weights=x["n"]),
                   "total_n": x["n"].sum()
               }))
               .reset_index(drop=True)
)

pivot_cat_mat = pivot_category_channel.pivot(
    index=product_dim,
    columns="channel",
    values="weighted_actual_uplift"
)

print("\n==================== first_page_category × Channel Weighted Actual Uplift Pivot ====================")
display(pivot_cat_mat)

if pivot_cat_mat.shape[0] > 0 and pivot_cat_mat.shape[1] > 0:
    plt.figure(figsize=(9, 6))
    plt.imshow(pivot_cat_mat.fillna(0).values, aspect="auto")
    plt.colorbar(label="Weighted Actual Uplift")
    plt.xticks(range(len(pivot_cat_mat.columns)), pivot_cat_mat.columns, rotation=30)
    plt.yticks(range(len(pivot_cat_mat.index)), pivot_cat_mat.index)
    plt.title("first_page_category × Channel Weighted Actual Uplift")
    plt.show()

# ------------------------------------------------
# 9) Save outputs
# ------------------------------------------------
os.makedirs(BASE_PATH, exist_ok=True)

combo_path = os.path.join(BASE_PATH, "part4b_first_page_category_segment_channel_actual_uplift.csv")
strategy_path = os.path.join(BASE_PATH, "part4b_first_page_category_segment_channel_strategy_table.csv")
pivot_channel_segment_path = os.path.join(BASE_PATH, "part4b_channel_segment_weighted_uplift_pivot_from_first_page_category.csv")
pivot_category_channel_path = os.path.join(BASE_PATH, "part4b_first_page_category_channel_weighted_uplift_pivot.csv")

combo_report.to_csv(combo_path, index=False)
strategy_table.to_csv(strategy_path, index=False)
pivot_channel_segment.to_csv(pivot_channel_segment_path, index=False)
pivot_category_channel.to_csv(pivot_category_channel_path, index=False)

print("\n✅ PART 4B-5 complete")
print("-", combo_path)
print("-", strategy_path)
print("-", pivot_channel_segment_path)
print("-", pivot_category_channel_path)

# ------------------------------------------------
# 10) Interpretation Summary
# ------------------------------------------------
print("\n==================== Interpretation Summary ====================")
print(f"Product-like column used: {product_dim}")

if len(top_expand) > 0:
    r = top_expand.iloc[0]
    print(
        f"Most promising combination: {product_dim}={r[product_dim]}, "
        f"target_segment={r['target_segment']}, channel={r['channel']}, "
        f"actual_uplift={r['actual_uplift']:.4f}, action={r['action_label']}"
    )

if len(bottom_combo) > 0:
    r = bottom_combo.iloc[0]
    print(
        f"Least efficient combination: {product_dim}={r[product_dim]}, "
        f"target_segment={r['target_segment']}, channel={r['channel']}, "
        f"actual_uplift={r['actual_uplift']:.4f}, action={r['action_label']}"
    )